In [ ]:
import re
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from urllib.parse import unquote

import pandas as pd

### Пути нужно будет заменить на свои

In [3]:
PHOTO_DIR = Path("./prod-svoe-vino-strapi")
SITEMAP_XML = PHOTO_DIR / "wines_sitemap_d778a8e06a.xml"
CSV_FILE = Path("./strapi_output0709.csv")
EXT = {".webp", ".png", ".jpg", ".jpeg", ".jfif", ".avif"}
FMT = ("thumbnail_", "small_", "medium_", "large_")

In [4]:
df = pd.read_csv(CSV_FILE).drop_duplicates(subset=["Slug"]).reset_index(drop=True)
df["Сорт винограда"] = df["Сорт винограда"].fillna("Не указан")
print("уникальных карточек:", len(df))

уникальных карточек: 2103


In [ ]:
# sitemap: slug - имя файла
NS = {
    "sm": "http://www.sitemaps.org/schemas/sitemap/0.9",
    "im": "http://www.google.com/schemas/sitemap-image/1.1",
}
slug2fname = {}
for url in ET.parse(SITEMAP_XML).getroot().findall("sm:url", NS):
    loc, img = url.find("sm:loc", NS), url.find("im:image/im:loc", NS)
    if loc is None or img is None or not (loc.text and img.text):
        continue
    slug2fname.setdefault(
        loc.text.strip().rstrip("/").split("/")[-1],
        unquote(img.text.strip().split("/")[-1]),
    )
print("slug в sitemap:", len(slug2fname))

slug в sitemap: 2037


In [ ]:
files = [p for p in PHOTO_DIR.rglob("*") if p.suffix.lower() in EXT]
by_name = {p.name: p for p in files}
by_name_low = {p.name.lower(): p for p in files}
print("файлов:", len(files))

файлов: 15760


In [ ]:
def locate(slug):
    fname = slug2fname.get(slug)
    if fname is None:
        return None, "no_sitemap"
    p = by_name.get(fname) or by_name_low.get(fname.lower())
    return (str(p), "ok") if p else (None, "no_file")


df[["photo_path", "match"]] = df["Slug"].apply(lambda s: pd.Series(locate(s)))

In [ ]:
def media_key(name):
    s = str(name).lower()
    s = re.sub(r"\.[a-z0-9]+$", "", s)
    for pre in FMT:
        if s.startswith(pre):
            s = s[len(pre) :]
            break
    s = re.sub(r"_[0-9a-f]{10}$", "", s)
    return re.sub(r"[^a-z0-9а-яё]+", "", s)


mk = {}
for p in files:
    k = media_key(p.name)
    cur = mk.get(k)
    if cur is None or (
        not p.name.lower().startswith(FMT) and cur.name.lower().startswith(FMT)
    ):
        mk[k] = p

for i in df.index[df["match"] != "ok"]:
    p = mk.get(media_key(df.at[i, "Название фото"])) or mk.get(
        media_key(df.at[i, "Slug"])
    )
    if p is not None:
        df.at[i, "photo_path"], df.at[i, "match"] = str(p), "ok_fallback"

df["tier"] = df["match"].map({"ok": 1, "ok_fallback": 2}).fillna(0).astype(int)
df["is_stale"] = df["photo_path"].isna()
df.loc[df["is_stale"], ["Slug", "Название вина"]].to_csv(
    "excluded_no_photo.csv", index=False
)
print(df["match"].value_counts())

match
ok             2037
ok_fallback      55
no_sitemap       11
Name: count, dtype: int64


In [ ]:
OUT = Path("./catalog_images")
OUT.mkdir(exist_ok=True)
for r in df.loc[df["photo_path"].notna(), ["Slug", "photo_path"]].itertuples(
    index=False
):
    p = Path(r.photo_path)
    shutil.copy2(p, OUT / f"{r.Slug}{p.suffix}")
df.to_csv("catalog_cleaned.csv", index=False)
print("галерея:", df["photo_path"].notna().sum(), "| stale:", int(df["is_stale"].sum()))

галерея: 2092 | stale: 11
